# 2. Analysing processed data with Raphael

A processed session: spikes from Raphael's processed tree, behaviour from this project.
Raphael loads the neurons and scores their tuning; the project says what the arena is.

This notebook uses a **demo session** (four place cells, two untuned units, one that failed curation).
For your own data, set `ROOT_DATA` and `session` to a session in your processed tree.

In [ ]:
import os
from pathlib import Path

# Run from the repository root, wherever the notebook was opened from.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)

import matplotlib.pyplot as plt

In [ ]:
from raphael.ephys.layout import Session

from starter_project.demo import make_processed_session

ROOT_DATA = Path('data/processed')
session = Session(mouse='1', day='1', session_type='OF1')
make_processed_session(ROOT_DATA, session)
sorted(p.name for p in session.folder(ROOT_DATA).iterdir())

## Load the neurons

`population.load` takes the files' shared prefix and returns the same object for spikes
(`_clusters.npz`) or calcium (`_traces.npz` + `_rois.csv`). `.accepted()` keeps units that passed curation.

In [ ]:
from raphael import population

pop = population.load(session.folder(ROOT_DATA) / f'{session.stem}_srt-kilosort4')
print(len(pop.ids), 'units,', int(pop.table['accepted'].sum()), 'accepted')
pop = pop.accepted()
pop.table

In [ ]:
rates = pop.binned(0.1)            # time x neuron TsdFrame, Hz
plt.figure(figsize=(8, 2.5))
plt.imshow(rates.values[:600].T, aspect='auto', cmap='magma',
           extent=[0, 60, len(pop.ids) - 0.5, -0.5])
plt.xlabel('s'); plt.ylabel('unit'); plt.colorbar(label='Hz'); plt.title('first minute');

## Behaviour (project code)

How behaviour is stored is the project's choice; `starter_project.behaviour` loads it into the
pynapple objects Raphael's scores read: `P_x`, `P_y`, `S`, `H` and a `moving` epoch.

In [ ]:
from starter_project.behaviour import load_behaviour

beh = load_behaviour(ROOT_DATA, session)
plt.figure(figsize=(3.5, 3.5))
plt.plot(beh['P_x'].values, beh['P_y'].values, lw=0.2, color='0.5')
plt.gca().set_aspect('equal'); plt.title('path'); plt.xlabel('cm'); plt.ylabel('cm');

## Rate maps and tuning scores

`rate_maps` and `score_session` (in `starter_project/analysis.py`) call Raphael's
tuning scores with this project's `bin_config` (`starter_project/arena.py`).
Each score is tested against a shuffled null.

In [ ]:
from starter_project.analysis import rate_maps, score_session

maps = rate_maps(ROOT_DATA, session)
scores = score_session(ROOT_DATA, session, n_shuffles=50)

fig, axes = plt.subplots(1, len(maps), figsize=(2.2 * len(maps), 2.4))
for ax, row in zip(axes, scores.itertuples()):
    ax.imshow(maps[row.unit].T, origin='lower', cmap='viridis')
    ax.set_title(f'unit {row.unit}\nSI {row.spatial_information:.2f}' + (' *' if row.si_sig else ''),
                 fontsize=8)
    ax.axis('off')
scores

## Using Raphael's functions directly

The same scores, one unit at a time, from `raphael.tuning_scores`:

In [ ]:
from raphael.tuning_scores import compute_spatial_information

from starter_project.arena import bin_config

unit = pop.ids[0]
result, = compute_spatial_information(beh, bin_config(), pop.data[unit], unit,
                                      alpha=0.05, n_shuffles=50)
plt.figure(figsize=(4, 2.5))
plt.hist(result['null_distribution'], bins=20, color='0.7', label='shuffles')
plt.axvline(result['spatial_information'], color='r', label=f'unit {unit}')
plt.xlabel('spatial information (bits/spike)'); plt.legend();

## Next

- Put analysis you want to keep in `src/starter_project/` and call it from a script in `scripts/`
  (see `scripts/analyse_tuning.py`), so figures can be rebuilt from the command line.
- Something other projects would use too? Suggest it for Raphael.